In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import glob
import os

In [ ]:
def interpolate_row(row):
    """a method for interpolating a row to 100 values

    Parameters:
    row (dataframe record): the row to be interpolated to 100 frames

    Returns:
    interpolated_row (dataframe record): the interpolated row
    """
    non_nan_indices = np.where(~np.isnan(row))[0]
    if len(non_nan_indices) <= 1:
        return np.full(100, np.nan)
    interpolated_row = np.interp(
        np.linspace(0, 1, 100),
        np.linspace(0, 1, len(non_nan_indices)),
        row[non_nan_indices],
    )
    return interpolated_row

In [ ]:
def interpolate(df):
    data_array = df.to_numpy()
    interpolated_data = np.apply_along_axis(interpolate_row, axis=1, arr=data_array)
    X_data = pd.DataFrame(interpolated_data)
    return X_data

In [ ]:
def create_plot_data(folder_path):
    print("Working on file", folder_path)
    df = pd.read_csv(os.path.join(folder_path, "merged_gait_count_annotations.csv"))

    # Keep only relevant columns
    df = df[['stepcount', 'jointAngleXZY_jLeftKnee_z', 'walk_mode']]

    output_folder = os.path.join(folder_path, "annotation_gaits_divided_by_label")
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Group by `walk_mode`
    for label, group_df in df.groupby('walk_mode'):
        # Pivot the dataframe to make 'stepcount' the index, and 'jointAngleXZY_jRightKnee_z' values in columns
        df_pivot = group_df.pivot_table(index='stepcount', values='jointAngleXZY_jLeftKnee_z', aggfunc=list)

        # Convert the lists into separate columns by creating a new DataFrame
        df_expanded = pd.DataFrame(df_pivot['jointAngleXZY_jLeftKnee_z'].tolist(), index=df_pivot.index)
        
        # Interpolate the expanded dataframe
        df_interpolated = interpolate(df_expanded)

        # Save the interpolated dataframe to a CSV file with the `walk_mode` label in the filename
        filename = f"{output_folder}\\gait_annotations_interpolated_{label}.csv"
        df_interpolated.to_csv(filename, index=False)

In [ ]:
dataset_path = "data_set"
print("Creating plot_data...")
for course_folder in os.listdir(dataset_path):
    course_folder_path = os.path.join(dataset_path, course_folder)
    if os.path.isdir(course_folder_path):
        for subfolder in os.listdir(course_folder_path):
            subfolder_path = os.path.join(course_folder_path, subfolder)
            if os.path.isdir(subfolder_path):
                create_plot_data(subfolder_path)
print("Plot data created successfully")

In [ ]:
def plot_combined_gait_cycles_by_file_type(base_folder):
    """
    Plots gait cycle data for predefined file names across all participants and courses.
    Each subplot combines data from all participants for the same file name.
    """
    # List of specific file names to search for
    file_names = [
        "gait_annotations_interpolated_slope_down.csv",
        "gait_annotations_interpolated_slope_up.csv",
        "gait_annotations_interpolated_stairs_down.csv",
        "gait_annotations_interpolated_stairs_up.csv",
        "gait_annotations_interpolated_walk.csv"
    ]
    
    # Create a subplot for each file type
    num_files = len(file_names)
    fig, axes = plt.subplots(nrows=num_files, ncols=1, figsize=(10, 3 * num_files), sharex=True)
    
    # Ensure axes is iterable even for a single subplot
    if num_files == 1:
        axes = [axes]
    
    # Iterate through each file name to plot its data
    for i, file_name in enumerate(file_names):
        # Gather all paths for the current file name across the folder structure
        search_pattern = os.path.join(base_folder, "course*", "id*", "annotation_gaits_divided_by_label", file_name)
        file_paths = glob.glob(search_pattern)
        
        if not file_paths:
            axes[i].set_title(f"No data for {file_name}")
            axes[i].set_ylabel("Amplitudes")
            continue  # Skip if no files found for this type
        
        # Plot all files of the current type in the same subplot
        for file_path in file_paths:
            df = pd.read_csv(file_path)
            # Plot each row (gait cycle) on the same subplot
            for gait in df.index:
                axes[i].plot(df.columns, df.loc[gait], label=os.path.basename(os.path.dirname(file_path)), alpha=0.6)
        
        # Set titles and labels for the subplot
        axes[i].set_title(f"Combined Data for {file_name}")
        axes[i].set_ylabel("Amplitudes")
        axes[i].tick_params(axis='x', rotation=90)  # Rotate x-axis labels vertically

    # Add common x-axis label and main title
    fig.supxlabel("Frames")
    fig.suptitle("Combined Gait Cycles Across All Participants by File Type", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.95])  # Leave space for main title
    plt.show()


In [ ]:
plot_combined_gait_cycles_by_file_type("data_set")